# Reality Bites

Part 8 built a RAG system that worked beautifully -- but it worked on a book: one document, written by one author, clean structure throughout. Most real corpora don't look like that. They're scattered across different systems, in inconsistent formats, full of internal jargon and administrative shorthand.

This part builds a bot that helps a student choose a TAF (Thématique d'Approfondissement) at IMT Atlantique, mixing RAG -- grounded in real course descriptions, not invented ones -- with genuine agentic decision-making. The corpus itself is real too: 279 actual TAF fiches, and once we index them with the exact same code that worked so well on the book, retrieval quality drops noticeably. Diagnosing *why*, and fixing it, is where most of the real work in a RAG system actually lives.

In [ ]:
# Program 1: recreate the toolkit from Part 8 -- local embedding model, chunker, retrieval

import os
import re
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from dotenv import load_dotenv

load_dotenv(override=True)

embed_name = "intfloat/multilingual-e5-small"
print(f"Loading {embed_name}...")
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()
print(f"  loaded: {embed_model.config.num_hidden_layers} layers, "
      f"{embed_model.config.hidden_size}-dimensional embeddings")

def embed(texts, batch_size=32):
    """Same mean-pooling as Part 7/8, plus length-1 normalisation, batched so a few hundred
    passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

def chunk_text(text, size=180, overlap=40):
    """Cut text into overlapping passages of `size` words -- Part 8's fixed-size chunker."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

def is_useful(chunk):
    """Drop chunks that are mostly punctuation and page numbers, not real text."""
    letters = sum(character.isalpha() for character in chunk)
    return letters / max(len(chunk), 1) > 0.6

def retrieve(question, chunks, vectors, top_k=3):
    question_vector = embed_query(question)
    scores = vectors @ question_vector          # one dot product per chunk, in one operation
    best = scores.topk(top_k)
    return [(scores[i].item(), chunks[i]) for i in best.indices.tolist()]

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "documents"))

# Self-test: embed a couple of unrelated passages plus one query, and confirm retrieve()
# actually finds the right one -- proof the toolkit works before the next programs lean on it.
test_chunks = ["The weather in Rennes is often rainy.", "Python is a popular programming language."]
test_vectors = embed_passages(test_chunks)
test_score, test_chunk = retrieve("what language is used for coding?", test_chunks, test_vectors, top_k=1)[0]
print(f"Self-test: {test_score:.3f} similarity, retrieved {test_chunk!r}")
print("Local embedding model and chunking helpers ready.")

## The corpus: real TAF fiches from PASS

Unlike the book, nobody hands us a single tidy PDF for this. IMT Atlantique's actual TAF catalogue lives in **PASS**, the school's internal course-management system, behind a Shibboleth SSO login that only staff and students can reach -- not something a web-search agent can just find and download.

So this corpus wasn't collected by an agent inside this notebook. It comes from `browse_pass.py`, a small standalone script sitting alongside this notebook -- and deliberately **not agentic at all**: it opens a real Chrome window with Playwright, waits for a human to type their SSO password, then walks PASS's "Catalogue UE - TAF" page, clicking every course entry, capturing the popup window that opens for each one (across all its frames, since PASS still uses old-school `<frameset>`s), and saving the result as a `.txt` file. No LLM decides anything in it -- it's browser automation with a fixed script, run once by hand rather than as a repeatable notebook cell, because it needs a real SSO login that can't be scripted into a reproducible exercise.

What it leaves behind is reproducible, though: 279 fiches already sitting in `documents/taf/`.

In [ ]:
# Program 2: the TAF fiches already collected from PASS via browse_pass.py

TAF_DIR = os.path.join(DOCS_DIR, "taf")
taf_files = sorted(f for f in os.listdir(TAF_DIR) if f.endswith(".txt"))
print(f"{len(taf_files)} fiches in documents/taf/\n")

print("A few of them:")
for filename in taf_files[:5]:
    print(f"  {filename}")
print("  ...")

print("\nOne fiche, to see the shape of the data:")
with open(os.path.join(TAF_DIR, taf_files[0])) as f:
    print(f.read()[:600])

Each fiche keeps the metadata `browse_pass.py` captured alongside the content -- which TAF-choice menu it came from (`CHOIX_TAF`), the course title (`TITRE_FICHE`), and the PASS URL it was popped up from -- then the raw page text underneath, frame by frame exactly as PASS rendered it: course code, responsable(s), équipe pédagogique, credits, site, and so on.

`documents/taf/` stays local -- not committed, the same way `.env` isn't -- but it's what every program below actually indexes.

## The same code, a much worse result

Now let's index this new corpus with *exactly* the code from Program 1 -- same chunking, same junk filter, same embedding model -- and ask it the kind of question a student would actually ask.

In [ ]:
# Program 3.1: index the collected documents with the very same code -- and watch it struggle

def load_documents(directory, chunker):
    """Read every .txt file in a directory, remembering which file each chunk came from."""
    chunks, sources = [], []
    for filename in sorted(f for f in os.listdir(directory) if f.endswith(".txt")):
        text = open(os.path.join(directory, filename)).read()
        for chunk in chunker(text):
            if is_useful(chunk):
                chunks.append(chunk)
                sources.append(filename)
    return chunks, sources

taf_chunks, taf_sources = load_documents(TAF_DIR, chunk_text)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks from {len(set(taf_sources))} documents\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Compare that with the book. There, each question landed on the passage that answered it. Here the top hits are vague -- a page header, a run of programme acronyms, a chunk about medical imaging when you asked about connected devices. The catalogue really does contain a `DASCI – DATA SCIENCE` record and an `IOT – INTERNET OF THINGS` one, and neither comes back.

Nothing is broken: same model, same code, same junk filter. What changed is the **shape of the documents**.

The book is continuous prose. Cut it anywhere and you still get a passage about one topic, because that's how prose works -- a paragraph about 6LoWPAN is surrounded by more text about 6LoWPAN.

A course catalogue is the opposite: a **list of short, self-contained records**, one per programme, each only a dozen lines long. Cutting every 180 words pays no attention to those boundaries, so one chunk ends up holding the tail of one TAF, all of the next, and the start of a third. Its embedding is the average of three unrelated programmes -- close to nothing in particular -- while a question about exactly one of them has nothing precise to match.

The fix isn't a better model or a bigger chunk. It's to **cut where the document says to cut**: the catalogue marks every record with a heading of its own,

```
CYBER – CYBERSECURITY (4+3)
DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER (3+4)
IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0 (3+3)
```

so we split on those instead, and each chunk becomes exactly one programme. Fixed-size chunking stays the fallback for documents with no such structure -- like the book.

In [ ]:
# Program 3.2: cut on the document's own section headings instead of every 180 words

# A TAF record always starts with a heading like "CYBER – CYBERSECURITY (4+3)":
# an acronym, a dash, a title, and the number of semesters.
SECTION_HEADING = re.compile(r"^\s*[A-Z][A-Z0-9&*\-' ]{1,40}\s*[–-]\s+.{3,70}\(\d\+\d\)\s*$")

def chunk_structured(text):
    """Split on section headings when the document has them, fixed-size otherwise."""
    lines = text.splitlines()
    headings = [i for i, line in enumerate(lines) if SECTION_HEADING.match(line)]

    if len(headings) < 3:            # no real structure -- the book takes this path
        return chunk_text(text)

    chunks = []
    if headings[0] > 0:              # whatever comes before the first heading
        chunks += chunk_text("\n".join(lines[:headings[0]]))
    for start, end in zip(headings, headings[1:] + [len(lines)]):
        section = "\n".join(lines[start:end]).strip()
        # One record per chunk -- unless a record is itself too long to embed intact.
        chunks += chunk_text(section) if len(section.split()) > 180 else [section]
    return chunks

taf_chunks, taf_sources = load_documents(TAF_DIR, chunk_structured)
taf_vectors = embed_passages(taf_chunks)
print(f"{len(taf_chunks)} chunks, one per programme wherever the document allowed it\n")

for question in ["Which TAF is about data science and artificial intelligence?",
                 "I want to work on connected devices and industry 4.0"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, taf_chunks, taf_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:110]}...")
    print()

Both questions now land on the right record -- `DASCI – DATA SCIENCE: FROM DATA TO DECISION-MAKER` and `IOT – INTERNET OF THINGS FOR THE INDUSTRY 4.0` -- where Program 3.1 surfaced neither. Same corpus, same embedding model, same retrieval code. **The only thing that changed is where we cut the text**, and it was worth more than any model upgrade would have been. (While writing this part we tried a bigger embedding model: it made no difference. Chunking did.)

Two caveats worth keeping in mind, because they're the normal condition of RAG rather than defects of this example:

* **Your results will differ from ours.** The librarian collects whatever the web offers today, so your corpus isn't ours. A question whose TAF happens to be missing from the documents you collected will still come back with something vaguely related -- retrieval always returns its closest matches, even when nothing is genuinely close. Being unable to say "I don't know" is a real limitation of plain similarity search.
* **The heading pattern is specific to these catalogues.** `SECTION_HEADING` matches how IMT Atlantique formats a TAF record; another corpus needs another rule -- Markdown `##` headings, numbered clauses, `<h2>` tags. There's no universal chunker, which is exactly why chunking deserves this much attention.

## A TAF advisor you can actually talk to

Wire `taf_chunks` into a `@function_tool` exactly like `search_book` in Part 8's Program 3, and you have Part 6's TAF advisor again -- except its knowledge now comes from real documents you can open and check, rather than descriptions written by hand. Wrap that in a Gradio `ChatInterface`, same pattern as Part 2, and it's no longer a notebook cell you re-run with a different string -- it's something you can actually have a conversation with.

In [ ]:
# Program 4: a TAF advisor over taf_chunks, wrapped in a Gradio chat interface

from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import gradio

# Gemini here, not Rennes: same OpenAI-compatible pattern as always, pointed at a
# different provider -- the portability Part 1 introduced, now actually paying off
# since Rennes' API has been down since this part was written. The Flash-Lite variant,
# not the full Flash model: its free tier allows far more requests per day, which
# matters for a notebook meant to be run and re-run by an entire class.
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=os.environ["GOOGLE_API_KEY"])
gemini_model = OpenAIChatCompletionsModel(model="gemini-flash-lite-latest", openai_client=gemini_client)

@function_tool
def search_taf(question: str):
    """Search the TAF documentation for the passages most relevant to a question."""
    print(f"[search_taf] the model asked: {question!r}")
    passages = retrieve(question, taf_chunks, taf_vectors)
    for score, chunk in passages:
        print(f"[search_taf]   -> {score:.3f}  {' '.join(chunk.split())[:100]}...")
    return "\n\n---\n\n".join(chunk for _, chunk in passages)

taf_advisor = Agent(
    name="TAF Advisor",
    instructions="Answer questions about IMT Atlantique's TAF programs using the search_taf "
                 "tool. Base your answer only on the passages it returns -- if they don't "
                 "contain the answer, say so rather than inventing one.",
    model=gemini_model,
    tools=[search_taf],
)

async def chat_async(message, history):
    try:
        result = await Runner.run(taf_advisor, message, max_turns=6)
        return result.final_output
    except Exception as e:
        return f"Error: {e}"

gradio.ChatInterface(chat_async, title="TAF Advisor").launch()

Run the cell above and a chat window opens (Gradio prints the local URL if it doesn't open one for you). Every message you type calls `chat_async`, which runs `taf_advisor`, which calls `search_taf` -- watch the `[search_taf]` trace printed above as you chat, to see what the model actually searched for and what came back.

## Key takeaways

<img src="images/takeaways.jpg" width="150" alt="Key takeaways" style="float: left; margin-right: 15px; margin-bottom: 10px;">

* **Real corpora are messier than clean documents.** A book is continuous prose, written by one author with consistent structure; a website is scattered across pages, formats and navigation cruft nobody asked for.
* **Chunking must match the document's actual shape.** Fixed-size chunking works on continuous prose because meaning doesn't care where you cut it. It fails on a list of short, self-contained records (like a course catalogue), because cutting every 180 words ignores the boundaries between them -- a chunk ends up mixing three unrelated topics into one meaningless average.
* **The fix is to cut where the document says to cut** -- on its own headings or structure -- rather than reaching for a bigger or different model. We tried a bigger embedding model while writing this part: no difference. Chunking was the whole story.
* **Extraction quality matters just as much as chunking.** A scraped web page is mostly navigation; menus repeated on every page are generic enough to rank against every question, crowding out passages that are actually about one topic. Strip that out at extraction time.
* **Retrieval can't say "I don't know".** It always returns its closest matches, however far away they are -- so a corpus missing the answer produces confident-looking passages about something else.
* When retrieval disappoints, look at your **documents** before reaching for a bigger model. Ours was a chunking problem and an extraction problem -- neither of which a better embedding model would have solved.
* A retrieval tool becomes an actual product the moment it's wrapped in a chat interface (Part 2's `gradio.ChatInterface` pattern) instead of a notebook cell you re-run by hand.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `AutoModel.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool` -- all introduced in Part 7):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `vectors @ query` (`torch`) | a matrix of chunk vectors, one query vector | one similarity score per chunk, in a single operation | Programs 1, 3.1, 3.2, 4 |
| `tensor.topk(k)` (`torch`) | how many results to keep | the k highest scores and their indices | Programs 1, 3.1, 3.2, 4 |
| `PdfReader(path_or_bytes)` (`pypdf`) | a file path, or a `BytesIO` of downloaded bytes | a PDF whose `.pages` expose `.extract_text()` | Program 2 |
| `soup.find("main")` (`bs4`) | a tag name | the first matching element, or `None` -- used to skip navigation | Program 2 |
| `gradio.ChatInterface(fn).launch()` (`gradio`) | an async function taking `(message, history)` | a running local chat server | Program 4 |

## Managed alternatives

Everything in Parts 7 to 9 was built by hand -- deliberately, since seeing the moving parts is the whole point of a course. In practice, plenty of providers now sell this same pipeline as a finished product: Google's **NotebookLM**, where you upload documents through a web page and it handles chunking, embedding, retrieval and citations entirely on its own; OpenAI's `file_search` tool with vector stores; and similar offerings from most major LLM providers. They save the work this notebook just did by hand, at the cost of the control over it -- when a corpus turns out to need something specific, like Program 3.2's structured chunking, a managed service either already handles it or it doesn't, and there's no code to open and fix. Knowing what's actually happening underneath, which is what these three parts were for, is what lets you tell the difference and build your own when the managed version doesn't fit.